# Medical LLM Notebook 8 — Explainable Fuzzy Hallucination-Risk Fusion

## Purpose
This notebook integrates the graph and hypergraph signals identified in Notebooks 6B and 7 into an interpretable fuzzy risk system.

### Fixed four-feature system
The preregistered/exploratory candidate set entering the fuzzy stage is:

**Graph features**
- Max GCCDC
- Max Structural Influence

**Hypergraph features (primary k=3)**
- Overlap Pair Fraction
- Hyperdegree Gini

For every outer cross-validation fold, response-length normalization is **refit using training data only**. Feature orientation and fuzzy membership scaling are also learned from training data only.

### Fuzzy inference
For each oriented, length-normalized feature, a high-risk membership is defined by a logistic membership function on a robustly standardized residual:

\[
\mu_H(z)=\frac{1}{1+e^{-z}}, \qquad \mu_L(z)=1-\mu_H(z).
\]

The two graph memberships are averaged to form graph risk evidence \(G\), and the two hypergraph memberships are averaged to form hypergraph evidence \(H\).

A zero-order Sugeno rule base is then used:

- Graph Low AND Hypergraph Low → 0.10
- Graph High AND Hypergraph Low → 0.60
- Graph Low AND Hypergraph High → 0.60
- Graph High AND Hypergraph High → 0.90

The final fuzzy risk is the normalized weighted average of these four rules.

### Validation strategy
- 10 repetitions of 5-fold pair-grouped cross-validation
- folds stratified by hallucination difficulty at the **Pair_ID** level
- train-only length normalization
- train-only orientation and membership calibration
- train-only decision-threshold calibration for thresholded metrics
- graph-only and hypergraph-only fuzzy evidence baselines
- logistic regression reference baseline using the same four normalized inputs
- leakage-aware nested feature-selection sensitivity analysis

**Important:** The four fixed features were discovered on the same 100-pair cohort in prior exploratory notebooks. Therefore fixed-model performance is internal/exploratory validation, not independent external validation. The nested sensitivity analysis partially addresses this by reselecting features using training folds only.

In [ ]:
import os, glob, zipfile, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon, rankdata, spearmanr
from statsmodels.stats.multitest import multipletests

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    balanced_accuracy_score, accuracy_score,
    f1_score, matthews_corrcoef, confusion_matrix,
    brier_score_loss, roc_curve
)

warnings.filterwarnings("ignore")

SEED = 42
REPEATS = 10
N_SPLITS = 5
PRIMARY_K = 3

print("Notebook 8 libraries loaded.")

## 1. Locate or upload Notebook 5 and Notebook 7 output ZIP files

In [ ]:
NB5_ZIP = "/mnt/data/Notebook5_100Pair_Outputs.zip"
NB7_ZIP = "/mnt/data/Notebook7_Hypergraph_Outputs.zip"

if not (os.path.exists(NB5_ZIP) and os.path.exists(NB7_ZIP)):
    try:
        from google.colab import files
        uploaded = files.upload()
        zip_names = [x for x in uploaded if x.lower().endswith(".zip")]
        assert len(zip_names) >= 2, "Upload Notebook5_100Pair_Outputs.zip and Notebook7_Hypergraph_Outputs.zip"
        nb5_candidates = [x for x in zip_names if "Notebook5" in x or "100Pair" in x]
        nb7_candidates = [x for x in zip_names if "Notebook7" in x or "Hypergraph" in x]
        assert nb5_candidates and nb7_candidates, "Could not identify both ZIP files."
        NB5_ZIP = nb5_candidates[0]
        NB7_ZIP = nb7_candidates[0]
    except ImportError:
        raise FileNotFoundError("Place Notebook5_100Pair_Outputs.zip and Notebook7_Hypergraph_Outputs.zip in the working directory.")

print("Notebook 5:", NB5_ZIP)
print("Notebook 7:", NB7_ZIP)

## 2. Load and merge raw graph and primary-k hypergraph features

In [ ]:
WORK = "/tmp/Notebook8_input"
os.makedirs(WORK, exist_ok=True)

for zpath, sub in [(NB5_ZIP, "nb5"), (NB7_ZIP, "nb7")]:
    dest = os.path.join(WORK, sub)
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall(dest)

response_file = glob.glob(WORK + "/nb5/**/MedHallu_200_Response_Features.csv", recursive=True)[0]
hyper_file = glob.glob(WORK + "/nb7/**/Hypergraph_Response_Features_AllK.csv", recursive=True)[0]

graph = pd.read_csv(response_file)
hyper_all = pd.read_csv(hyper_file)
hyper = hyper_all[hyper_all["K"] == PRIMARY_K].copy()

hyper_cols = [
    "Pair_ID", "Response_Type", "K",
    "Mean_Hyperedge_Weight", "Max_Hyperedge_Weight",
    "Mean_Hyperdegree", "Max_Hyperdegree",
    "Hyperdegree_Concentration", "Hyperdegree_Gini",
    "Overlap_Pair_Fraction", "Mean_Overlap_Jaccard",
    "TwoSection_Density", "Hypergraph_Lambda2",
    "Hypergraph_Spectral_Entropy"
]

data = graph.merge(
    hyper[hyper_cols],
    on=["Pair_ID", "Response_Type"],
    how="inner",
    validate="one_to_one"
)

assert len(data) == 200
assert data["Pair_ID"].nunique() == 100
assert data[["Pair_ID", "Response_Type"]].duplicated().sum() == 0
assert data.isna().sum().sum() == 0

print("Merged data:", data.shape)
print(data["Response_Type"].value_counts())

## 3. Feature sets

In [ ]:
FIXED_GRAPH = [
    "Max_GCCDC",
    "Max_Structural_Influence"
]

FIXED_HYPER = [
    "Overlap_Pair_Fraction",
    "Hyperdegree_Gini"
]

FIXED_FEATURES = FIXED_GRAPH + FIXED_HYPER

GRAPH_POOL = [
    "GCC", "Communication_Efficiency", "Mean_Edge_Weight",
    "Max_PageRank", "PageRank_Concentration",
    "Mean_Betweenness", "Max_Betweenness", "Max_GCCDC",
    "Mean_Structural_Influence", "Max_Structural_Influence",
    "QAS_Mean", "QAS_Median", "QAS_Std"
]

HYPER_POOL = [
    "Mean_Hyperedge_Weight", "Max_Hyperedge_Weight",
    "Mean_Hyperdegree", "Max_Hyperdegree",
    "Hyperdegree_Concentration", "Hyperdegree_Gini",
    "Overlap_Pair_Fraction", "Mean_Overlap_Jaccard",
    "TwoSection_Density", "Hypergraph_Lambda2",
    "Hypergraph_Spectral_Entropy"
]

ALL_POOL = GRAPH_POOL + HYPER_POOL

print("Fixed graph:", FIXED_GRAPH)
print("Fixed hypergraph:", FIXED_HYPER)

## 4. Length normalization and fuzzy helper functions

In [ ]:
def fit_length_models(train_df, test_df, features, model_type="spline"):
    keep = ["Pair_ID", "Response_Type", "Label", "Difficulty", "Hallucination_Category", "Words"]
    train_norm = train_df[keep].copy().reset_index(drop=True)
    test_norm = test_df[keep].copy().reset_index(drop=True)

    x_train = np.log1p(train_df["Words"].astype(float).to_numpy()).reshape(-1, 1)
    x_test = np.log1p(test_df["Words"].astype(float).to_numpy()).reshape(-1, 1)

    for feature in features:
        y_train = train_df[feature].astype(float).to_numpy()

        if model_type == "spline":
            model = make_pipeline(
                SplineTransformer(n_knots=4, degree=3, include_bias=False),
                LinearRegression()
            )
        elif model_type == "linear":
            model = LinearRegression()
        else:
            raise ValueError("model_type must be spline or linear")

        model.fit(x_train, y_train)
        pred_train = model.predict(x_train)
        pred_test = model.predict(x_test)

        residual_train = y_train - pred_train
        scale = np.std(residual_train, ddof=1)
        if not np.isfinite(scale) or scale <= 1e-12:
            scale = 1.0

        train_norm[feature] = residual_train / scale
        test_norm[feature] = (
            test_df[feature].astype(float).to_numpy() - pred_test
        ) / scale

    return train_norm, test_norm


def robust_center_scale(x):
    x = np.asarray(x, dtype=float)
    median = np.median(x)
    q1, q3 = np.quantile(x, [0.25, 0.75])
    scale = (q3 - q1) / 1.349
    if not np.isfinite(scale) or scale <= 1e-8:
        scale = np.std(x, ddof=1)
    if not np.isfinite(scale) or scale <= 1e-8:
        scale = 1.0
    return float(median), float(scale)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


def fit_fuzzy_parameters(train_norm, graph_features, hyper_features):
    params = {}
    for feature in graph_features + hyper_features:
        factual_mean = train_norm.loc[train_norm["Label"] == 0, feature].mean()
        hall_mean = train_norm.loc[train_norm["Label"] == 1, feature].mean()
        sign = 1.0 if hall_mean >= factual_mean else -1.0
        oriented = sign * train_norm[feature].astype(float).to_numpy()
        center, scale = robust_center_scale(oriented)
        params[feature] = {"sign": sign, "center": center, "scale": scale}
    return params


def apply_fuzzy_system(norm_df, params, graph_features, hyper_features):
    out = norm_df[["Pair_ID", "Response_Type", "Label", "Difficulty", "Hallucination_Category", "Words"]].copy()
    mu = {}

    for feature in graph_features + hyper_features:
        p = params[feature]
        z = (
            p["sign"] * norm_df[feature].astype(float).to_numpy() - p["center"]
        ) / p["scale"]
        mu[feature] = sigmoid(z)
        out[f"mu_high_{feature}"] = mu[feature]

    G = np.mean([mu[f] for f in graph_features], axis=0)
    H = np.mean([mu[f] for f in hyper_features], axis=0)

    GL, HL = 1.0 - G, 1.0 - H
    w_LL = GL * HL
    w_HL = G * HL
    w_LH = GL * H
    w_HH = G * H

    # Fixed, interpretable zero-order Sugeno consequents.
    fuzzy_risk = (
        0.10 * w_LL +
        0.60 * w_HL +
        0.60 * w_LH +
        0.90 * w_HH
    )

    rule_weights = np.vstack([w_LL, w_HL, w_LH, w_HH]).T
    rule_entropy = -np.sum(
        rule_weights * np.log(np.clip(rule_weights, 1e-12, 1.0)), axis=1
    ) / np.log(4.0)

    out["Graph_Risk"] = G
    out["Hypergraph_Risk"] = H
    out["Fuzzy_Risk"] = fuzzy_risk
    out["Rule_Entropy"] = rule_entropy
    return out


def best_training_threshold(y, score):
    values = np.unique(np.asarray(score, dtype=float))
    mids = (values[:-1] + values[1:]) / 2.0 if len(values) > 1 else np.array([0.5])
    candidates = np.unique(np.concatenate([[0.5], mids]))

    best_t, best_bacc = 0.5, -np.inf
    for threshold in candidates:
        pred = (np.asarray(score) >= threshold).astype(int)
        bacc = balanced_accuracy_score(y, pred)
        if (
            bacc > best_bacc + 1e-12 or
            (abs(bacc - best_bacc) <= 1e-12 and abs(threshold - 0.5) < abs(best_t - 0.5))
        ):
            best_t, best_bacc = float(threshold), float(bacc)
    return best_t


def signed_rank_biserial(delta):
    delta = np.asarray(delta, dtype=float)
    delta = delta[np.isfinite(delta)]
    delta = delta[delta != 0]
    if len(delta) == 0:
        return 0.0
    ranks = rankdata(np.abs(delta))
    w_pos = ranks[delta > 0].sum()
    w_neg = ranks[delta < 0].sum()
    return float((w_pos - w_neg) / (w_pos + w_neg)) if (w_pos + w_neg) else 0.0


def score_metrics(df, score_col, threshold_col):
    y = df["Label"].astype(int).to_numpy()
    score = df[score_col].astype(float).to_numpy()
    threshold = df[threshold_col].astype(float).to_numpy()
    pred = (score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "ROC_AUC": roc_auc_score(y, score),
        "PR_AUC": average_precision_score(y, score),
        "Balanced_Accuracy": balanced_accuracy_score(y, pred),
        "Accuracy": accuracy_score(y, pred),
        "F1": f1_score(y, pred),
        "MCC": matthews_corrcoef(y, pred),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Brier": brier_score_loss(y, score)
    }

## 5. Fixed four-feature repeated pair-grouped cross-validation

In [ ]:
pair_table = (
    data[["Pair_ID", "Difficulty"]]
    .drop_duplicates("Pair_ID")
    .sort_values("Pair_ID")
    .reset_index(drop=True)
)

prediction_blocks = []
fold_records = []

for repeat in range(REPEATS):
    splitter = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED + repeat
    )

    for fold, (train_idx, test_idx) in enumerate(
        splitter.split(pair_table["Pair_ID"], pair_table["Difficulty"])
    ):
        train_pairs = set(pair_table.iloc[train_idx]["Pair_ID"])
        test_pairs = set(pair_table.iloc[test_idx]["Pair_ID"])

        train = data[data["Pair_ID"].isin(train_pairs)].copy()
        test = data[data["Pair_ID"].isin(test_pairs)].copy()

        train_norm, test_norm = fit_length_models(
            train, test, FIXED_FEATURES, model_type="spline"
        )

        fuzzy_params = fit_fuzzy_parameters(
            train_norm, FIXED_GRAPH, FIXED_HYPER
        )

        train_risk = apply_fuzzy_system(
            train_norm, fuzzy_params, FIXED_GRAPH, FIXED_HYPER
        )
        test_risk = apply_fuzzy_system(
            test_norm, fuzzy_params, FIXED_GRAPH, FIXED_HYPER
        )

        # Reference logistic model on the same normalized four features.
        logistic = LogisticRegression(
            C=1.0, solver="liblinear", random_state=SEED + repeat
        )
        logistic.fit(train_norm[FIXED_FEATURES], train_norm["Label"])
        train_logistic = logistic.predict_proba(train_norm[FIXED_FEATURES])[:, 1]
        test_logistic = logistic.predict_proba(test_norm[FIXED_FEATURES])[:, 1]
        test_risk["Logistic_Risk"] = test_logistic

        thresholds = {
            "Fuzzy": best_training_threshold(train_risk["Label"], train_risk["Fuzzy_Risk"]),
            "Graph": best_training_threshold(train_risk["Label"], train_risk["Graph_Risk"]),
            "Hypergraph": best_training_threshold(train_risk["Label"], train_risk["Hypergraph_Risk"]),
            "Logistic": best_training_threshold(train_norm["Label"], train_logistic)
        }

        test_risk["Threshold_Fuzzy"] = thresholds["Fuzzy"]
        test_risk["Threshold_Graph"] = thresholds["Graph"]
        test_risk["Threshold_Hypergraph"] = thresholds["Hypergraph"]
        test_risk["Threshold_Logistic"] = thresholds["Logistic"]
        test_risk["Repeat"] = repeat
        test_risk["Fold"] = fold

        for feature in FIXED_FEATURES:
            test_risk[f"{feature}_Norm"] = test_norm[feature].to_numpy()
            test_risk[f"{feature}_Orientation"] = fuzzy_params[feature]["sign"]

        prediction_blocks.append(test_risk)
        fold_records.append({
            "Repeat": repeat,
            "Fold": fold,
            "Train_Pairs": len(train_pairs),
            "Test_Pairs": len(test_pairs),
            **{f"Threshold_{k}": v for k, v in thresholds.items()}
        })

fixed_predictions = pd.concat(prediction_blocks, ignore_index=True)
fold_info = pd.DataFrame(fold_records)

assert len(fixed_predictions) == 200 * REPEATS
print("Fixed-model out-of-fold prediction rows:", len(fixed_predictions))

## 6. Repeated-CV performance and baselines

In [ ]:
model_specs = {
    "Fuzzy": ("Fuzzy_Risk", "Threshold_Fuzzy"),
    "Graph": ("Graph_Risk", "Threshold_Graph"),
    "Hypergraph": ("Hypergraph_Risk", "Threshold_Hypergraph"),
    "Logistic": ("Logistic_Risk", "Threshold_Logistic")
}

metric_rows = []

for repeat, subset in fixed_predictions.groupby("Repeat"):
    for model_name, (score_col, threshold_col) in model_specs.items():
        m = score_metrics(subset, score_col, threshold_col)

        pair_scores = subset.pivot(
            index="Pair_ID", columns="Response_Type", values=score_col
        )
        delta = pair_scores["Hallucinated"] - pair_scores["Factual"]

        m.update({
            "Repeat": repeat,
            "Model": model_name,
            "Pairwise_Ordering_Accuracy": float(
                ((delta > 0).sum() + 0.5 * (delta == 0).sum()) / len(delta)
            ),
            "Mean_Paired_Risk_Delta": float(delta.mean()),
            "Median_Paired_Risk_Delta": float(delta.median())
        })
        metric_rows.append(m)

fixed_metrics = pd.DataFrame(metric_rows)

metric_summary = (
    fixed_metrics.groupby("Model")
    .agg(
        ROC_AUC_Mean=("ROC_AUC", "mean"), ROC_AUC_SD=("ROC_AUC", "std"),
        PR_AUC_Mean=("PR_AUC", "mean"), PR_AUC_SD=("PR_AUC", "std"),
        Balanced_Accuracy_Mean=("Balanced_Accuracy", "mean"), Balanced_Accuracy_SD=("Balanced_Accuracy", "std"),
        F1_Mean=("F1", "mean"), F1_SD=("F1", "std"),
        MCC_Mean=("MCC", "mean"), MCC_SD=("MCC", "std"),
        Sensitivity_Mean=("Sensitivity", "mean"), Sensitivity_SD=("Sensitivity", "std"),
        Specificity_Mean=("Specificity", "mean"), Specificity_SD=("Specificity", "std"),
        Pairwise_Ordering_Mean=("Pairwise_Ordering_Accuracy", "mean"),
        Pairwise_Ordering_SD=("Pairwise_Ordering_Accuracy", "std")
    )
    .reset_index()
)

display(metric_summary.round(4))

## 7. Representative out-of-fold paired risk analysis (repeat 0)

In [ ]:
representative = fixed_predictions[fixed_predictions["Repeat"] == 0].copy()

pair_risk = representative.pivot(
    index="Pair_ID", columns="Response_Type", values="Fuzzy_Risk"
)
pair_risk["Delta_HminusF"] = pair_risk["Hallucinated"] - pair_risk["Factual"]

paired_test = wilcoxon(
    pair_risk["Hallucinated"], pair_risk["Factual"],
    alternative="two-sided", zero_method="wilcox", method="auto"
)

paired_risk_summary = pd.DataFrame([{
    "Mean_Factual_Risk": pair_risk["Factual"].mean(),
    "Mean_Hallucinated_Risk": pair_risk["Hallucinated"].mean(),
    "Mean_Delta_HminusF": pair_risk["Delta_HminusF"].mean(),
    "Median_Delta_HminusF": pair_risk["Delta_HminusF"].median(),
    "Hallucinated_Higher_n": int((pair_risk["Delta_HminusF"] > 0).sum()),
    "Wilcoxon_W": paired_test.statistic,
    "Wilcoxon_p": paired_test.pvalue,
    "Rank_Biserial": signed_rank_biserial(pair_risk["Delta_HminusF"])
}])

display(paired_risk_summary.round(6))

## 8. Leakage-aware nested feature-selection sensitivity

In [ ]:
def feature_screen(train_norm, pool, top_n=2):
    factual = train_norm[train_norm["Response_Type"] == "Factual"].set_index("Pair_ID")
    hall = train_norm[train_norm["Response_Type"] == "Hallucinated"].set_index("Pair_ID")
    pair_ids = sorted(set(factual.index) & set(hall.index))

    delta_words = (
        hall.loc[pair_ids, "Words"].astype(float).to_numpy() -
        factual.loc[pair_ids, "Words"].astype(float).to_numpy()
    )

    rows = []
    for feature in pool:
        f = factual.loc[pair_ids, feature].astype(float).to_numpy()
        h = hall.loc[pair_ids, feature].astype(float).to_numpy()
        delta = h - f

        p = 1.0 if np.allclose(delta, 0) else wilcoxon(
            h, f, alternative="two-sided", zero_method="wilcox", method="auto"
        ).pvalue

        rho_length = spearmanr(train_norm["Words"], train_norm[feature]).statistic
        rho_delta_length = spearmanr(delta_words, delta).statistic

        rows.append({
            "Feature": feature,
            "p": p,
            "Rank_Biserial": signed_rank_biserial(delta),
            "Abs_Rank_Biserial": abs(signed_rank_biserial(delta)),
            "rho_length": rho_length,
            "rho_delta_length": rho_delta_length
        })

    table = pd.DataFrame(rows)
    table["p_Holm"] = multipletests(table["p"], method="holm")[1]
    table["Length_Robust"] = (
        table["rho_length"].abs() < 0.20
    ) & (
        table["rho_delta_length"].abs() < 0.20
    )

    table = table.sort_values(
        ["Length_Robust", "p_Holm", "Abs_Rank_Biserial"],
        ascending=[False, True, False]
    ).reset_index(drop=True)

    return table.head(top_n)["Feature"].tolist(), table


nested_prediction_blocks = []
selection_records = []

for repeat in range(REPEATS):
    splitter = StratifiedKFold(
        n_splits=N_SPLITS, shuffle=True, random_state=SEED + repeat
    )

    for fold, (train_idx, test_idx) in enumerate(
        splitter.split(pair_table["Pair_ID"], pair_table["Difficulty"])
    ):
        train_pairs = set(pair_table.iloc[train_idx]["Pair_ID"])
        test_pairs = set(pair_table.iloc[test_idx]["Pair_ID"])
        train = data[data["Pair_ID"].isin(train_pairs)].copy()
        test = data[data["Pair_ID"].isin(test_pairs)].copy()

        train_norm, test_norm = fit_length_models(
            train, test, ALL_POOL, model_type="spline"
        )

        selected_graph, graph_screen = feature_screen(train_norm, GRAPH_POOL, top_n=2)
        selected_hyper, hyper_screen = feature_screen(train_norm, HYPER_POOL, top_n=2)

        params = fit_fuzzy_parameters(train_norm, selected_graph, selected_hyper)
        train_risk = apply_fuzzy_system(train_norm, params, selected_graph, selected_hyper)
        test_risk = apply_fuzzy_system(test_norm, params, selected_graph, selected_hyper)

        threshold = best_training_threshold(
            train_risk["Label"], train_risk["Fuzzy_Risk"]
        )

        test_risk["Threshold_Fuzzy"] = threshold
        test_risk["Repeat"] = repeat
        test_risk["Fold"] = fold
        nested_prediction_blocks.append(test_risk)

        selection_records.append({
            "Repeat": repeat, "Fold": fold,
            "Graph_1": selected_graph[0], "Graph_2": selected_graph[1],
            "Hyper_1": selected_hyper[0], "Hyper_2": selected_hyper[1]
        })

nested_predictions = pd.concat(nested_prediction_blocks, ignore_index=True)
selection_table = pd.DataFrame(selection_records)

nested_metric_rows = []
for repeat, subset in nested_predictions.groupby("Repeat"):
    m = score_metrics(subset, "Fuzzy_Risk", "Threshold_Fuzzy")
    pair_scores = subset.pivot(index="Pair_ID", columns="Response_Type", values="Fuzzy_Risk")
    delta = pair_scores["Hallucinated"] - pair_scores["Factual"]
    m.update({
        "Repeat": repeat,
        "Model": "Nested_Fuzzy",
        "Pairwise_Ordering_Accuracy": float(
            ((delta > 0).sum() + 0.5 * (delta == 0).sum()) / len(delta)
        )
    })
    nested_metric_rows.append(m)

nested_metrics = pd.DataFrame(nested_metric_rows)

nested_summary = nested_metrics.agg({
    "ROC_AUC": ["mean", "std"],
    "PR_AUC": ["mean", "std"],
    "Balanced_Accuracy": ["mean", "std"],
    "F1": ["mean", "std"],
    "MCC": ["mean", "std"],
    "Sensitivity": ["mean", "std"],
    "Specificity": ["mean", "std"],
    "Pairwise_Ordering_Accuracy": ["mean", "std"]
}).T

display(nested_summary.round(4))

## 9. Nested feature-selection stability

In [ ]:
graph_selection_frequency = (
    pd.Series(selection_table[["Graph_1", "Graph_2"]].to_numpy().ravel())
    .value_counts()
    .rename_axis("Feature")
    .reset_index(name="Selections")
)
graph_selection_frequency["Selection_Frequency"] = (
    graph_selection_frequency["Selections"] / len(selection_table)
)

hyper_selection_frequency = (
    pd.Series(selection_table[["Hyper_1", "Hyper_2"]].to_numpy().ravel())
    .value_counts()
    .rename_axis("Feature")
    .reset_index(name="Selections")
)
hyper_selection_frequency["Selection_Frequency"] = (
    hyper_selection_frequency["Selections"] / len(selection_table)
)

print("Graph feature selection frequency")
display(graph_selection_frequency)
print("Hypergraph feature selection frequency")
display(hyper_selection_frequency)

## 10. Difficulty-level descriptive performance (representative repeat)

In [ ]:
difficulty_rows = []
for difficulty, subset in representative.groupby("Difficulty"):
    if subset["Label"].nunique() < 2:
        continue
    m = score_metrics(subset, "Fuzzy_Risk", "Threshold_Fuzzy")
    m["Difficulty"] = difficulty
    m["Responses"] = len(subset)
    m["Pairs"] = subset["Pair_ID"].nunique()
    difficulty_rows.append(m)

difficulty_performance = pd.DataFrame(difficulty_rows)
display(difficulty_performance.round(4))

## 11. Visual summaries

In [ ]:
# ROC curve — representative repeat
plt.figure(figsize=(6, 5))
for model_name, (score_col, _) in model_specs.items():
    fpr, tpr, _ = roc_curve(representative["Label"], representative[score_col])
    auc = roc_auc_score(representative["Label"], representative[score_col])
    plt.plot(fpr, tpr, label=f"{model_name} AUC={auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Notebook 8: Out-of-fold ROC curves (repeat 0)")
plt.legend()
plt.tight_layout()
ROC_FIG = "/tmp/Notebook8_ROC.png"
plt.savefig(ROC_FIG, dpi=200)
plt.show()

In [ ]:
# Paired factual vs hallucinated fuzzy-risk distributions
factual_risk = pair_risk["Factual"].to_numpy()
hall_risk = pair_risk["Hallucinated"].to_numpy()

plt.figure(figsize=(6, 5))
plt.boxplot([factual_risk, hall_risk], labels=["Factual", "Hallucinated"])
plt.ylabel("Fuzzy hallucination-risk score")
plt.title("Paired out-of-fold fuzzy risk (repeat 0)")
plt.tight_layout()
RISK_FIG = "/tmp/Notebook8_Paired_Risk.png"
plt.savefig(RISK_FIG, dpi=200)
plt.show()

## 12. Save all Notebook 8 outputs

In [ ]:
OUTPUT_DIR = "/tmp/Notebook8_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

fixed_predictions.to_csv(f"{OUTPUT_DIR}/Fixed_Fuzzy_OOF_Predictions_AllRepeats.csv", index=False)
fold_info.to_csv(f"{OUTPUT_DIR}/Fixed_Fuzzy_Fold_Information.csv", index=False)
fixed_metrics.to_csv(f"{OUTPUT_DIR}/Fixed_Fuzzy_Repeated_CV_Metrics.csv", index=False)
metric_summary.to_csv(f"{OUTPUT_DIR}/Fixed_Fuzzy_Metric_Summary.csv", index=False)
pair_risk.reset_index().to_csv(f"{OUTPUT_DIR}/Representative_Paired_Fuzzy_Risk.csv", index=False)
paired_risk_summary.to_csv(f"{OUTPUT_DIR}/Representative_Paired_Risk_Statistics.csv", index=False)
nested_predictions.to_csv(f"{OUTPUT_DIR}/Nested_Fuzzy_OOF_Predictions_AllRepeats.csv", index=False)
nested_metrics.to_csv(f"{OUTPUT_DIR}/Nested_Fuzzy_Repeated_CV_Metrics.csv", index=False)
selection_table.to_csv(f"{OUTPUT_DIR}/Nested_Feature_Selections_By_Fold.csv", index=False)
graph_selection_frequency.to_csv(f"{OUTPUT_DIR}/Nested_Graph_Feature_Selection_Frequency.csv", index=False)
hyper_selection_frequency.to_csv(f"{OUTPUT_DIR}/Nested_Hypergraph_Feature_Selection_Frequency.csv", index=False)
difficulty_performance.to_csv(f"{OUTPUT_DIR}/Difficulty_Performance_Repeat0.csv", index=False)

import shutil
shutil.copy(ROC_FIG, f"{OUTPUT_DIR}/ROC_Curves_Repeat0.png")
shutil.copy(RISK_FIG, f"{OUTPUT_DIR}/Paired_Fuzzy_Risk_Repeat0.png")

metadata = {
    "seed": SEED,
    "repeats": REPEATS,
    "folds": N_SPLITS,
    "primary_hypergraph_k": PRIMARY_K,
    "fixed_graph_features": FIXED_GRAPH,
    "fixed_hypergraph_features": FIXED_HYPER,
    "length_normalization": "train-only cubic spline on log1p(Words), standardized by training residual SD",
    "membership": "logistic high-risk membership after train-only orientation and robust scaling",
    "rule_consequents": {"LL": 0.10, "HL": 0.60, "LH": 0.60, "HH": 0.90},
    "validation_note": "Fixed four features were discovered on the same cohort; nested selection is included as a leakage-aware sensitivity analysis."
}
with open(f"{OUTPUT_DIR}/Notebook8_Metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

ZIP_OUT = "/tmp/Notebook8_Fuzzy_Risk_Outputs.zip"
with zipfile.ZipFile(ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in os.listdir(OUTPUT_DIR):
        z.write(os.path.join(OUTPUT_DIR, filename), arcname=filename)

print("Saved:", ZIP_OUT)
print("Output files:", len(os.listdir(OUTPUT_DIR)))

## Interpretation guidance

- Treat the fuzzy score as an **interpretable risk score**, not a calibrated clinical probability.
- Emphasize ROC-AUC, PR-AUC, pairwise ordering, and repeated-CV stability rather than a single accuracy value.
- The fixed four-feature model is an internal exploratory validation because the candidate features were identified on the same 100-pair cohort.
- The nested feature-selection analysis is more conservative because feature selection is repeated within training folds only.
- Do not describe attention or hypergraph features as causal explanations of hallucination.
- External validation on an independent medical hallucination dataset/model remains desirable before claiming generalizable detection performance.